# Models - finding a better way

This version is much like the Models notebook only this one is more to to standard.

In [1]:
'''
This isn't strictly needed. However it solves this annoying pandas error:


/opt/homebrew/Caskroom/miniforge/base/envs/supervised/lib/python3.9/site-packages/xgboost/compat.py:36: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index
  
The problem is solved with xgboost 1.6 but I don't want to use pip in this case and the conda package is currently 1.5.1  
'''

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb

from sklearn.pipeline import Pipeline, make_pipeline
from sklearn import preprocessing
from sklearn import set_config


from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.preprocessing import QuantileTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import MinMaxScaler

from sklearn.compose import TransformedTargetRegressor
from sklearn.compose import ColumnTransformer
from sklearn.compose import make_column_transformer

from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.decomposition import PCA

from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.feature_selection import SelectPercentile, chi2

from sklearn.metrics import r2_score, mean_squared_error,accuracy_score, make_scorer


from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

In [3]:
df = pd.read_pickle("../data/diamonds.pkl")
#df

### Reviewing the data

As we noted in the DataMunging notebook, this dataset has numerical, ordinal and categorical columns. So there is a little more work to be done in getting things ready.

In [4]:
df.dtypes

cut                             object
color                           object
clarity                         object
caret_weight                   float64
cut_quality                     object
lab                             object
symmetry                        object
polish                          object
eye_clean                       object
culet_size                      object
culet_condition                 object
depth_percent                  float64
table_percent                  float64
meas_length                    float64
meas_width                     float64
meas_depth                     float64
girdle_min                      object
girdle_max                      object
fluor_color                     object
fluor_intensity                 object
fancy_color_dominant_color      object
fancy_color_secondary_color     object
fancy_color_overtone            object
fancy_color_intensity           object
total_sales_price                int64
dtype: object

## Define the categories for transformations

We have numerical columns as well as categorical columns both of ordinal and nominal subtypes

#### Numerical 

In [5]:
numerical_columns = df.select_dtypes(include=['int64', 'float64']).columns.to_list()

numerical_columns

['caret_weight',
 'depth_percent',
 'table_percent',
 'meas_length',
 'meas_width',
 'meas_depth',
 'total_sales_price']

#### Ordinals

In [6]:
ordinal_columns = ['clarity', 'culet_size', 'cut_quality', 'polish', 'symmetry']

In [7]:
clarity_ord = ['F', 'IF', 'VVS1', 'VVS2', 'VS1', 'VS2', 'SI1',  'SI2', 'SI3', 'I1', 'I2','I3']
culet_size_ord = ['N', 'VS', 'S', 'M', 'SL', 'L', 'VL', 'EL', 'unknown']
cut_quality_ord = ['Ideal', 'Excellent',  'Very Good', 'Good', 'Fair', 'None', 'unknown']
polish_ord = ['Excellent','Very Good', 'Good', 'Fair', 'Poor']
symmetry_ord = ['Excellent','Very Good', 'Good', 'Fair', 'Poor']

#### Nominals

In [8]:
nominal_columns = ['cut', 'color','lab','eye_clean','culet_condition',\
                   'girdle_max', 'girdle_min',\
                   'fancy_color_intensity','fancy_color_dominant_color',\
                   'fancy_color_secondary_color','fancy_color_overtone',\
                   'fluor_color', 'fluor_intensity']
            
# for col in nominal_columns:
#     print(f" '{col}' has the following values: \n \t {df[col].unique()} \n")

In [9]:
# I don't need this but  like this piece of code
# all_categoricals = df.select_dtypes(exclude=np.number).columns.to_list()

## Preparing the model 

In [10]:
# split into inputs and outputs
X, y = df.drop('total_sales_price', axis=1), df['total_sales_price']


In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state=314)

In [12]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((153792, 24), (153792,), (65911, 24), (65911,))

## Make and run the pipeline

In [13]:
set_config(display='diagram')

In [ ]:
# nominal_pipeline = Pipeline(
#     steps=[("onehot", OneHotEncoder(handle_unknown="ignore")) ])
 
# clarity_pipeline = Pipeline(
#     steps=[("encoder", OrdinalEncoder()) ])

# culet_size_pipeline = Pipeline(
#     steps=[("encoder", OrdinalEncoder()) ])

# cut_quality_pipeline = Pipeline(
#     steps=[("encoder", OrdinalEncoder()) ])

# polish_pipeline = Pipeline(
#     steps=[("encoder", OrdinalEncoder()) ])

# symmetry_pipeline = Pipeline(
#     steps=[("encoder", OrdinalEncoder()) ])

In [ ]:
# col_trans = ColumnTransformer(transformers=
#     [   
#         ("nominal_cols", nominal_pipeline, ['nominal_columns']),
#         ("ordinal_cols", oridal)
        
        
# #         ("clarity", clarity_pipeline, ['clarity_ord']),
# #         ("culet_size", culet_size_pipeline, ["culet_size_ord"]),
# #         ("cut_quality", cut_quality_pipeline, ["cut_quality_ord"]),
# #         ("polish", polish_pipeline, ["polish_ord"]),       
# #         ("symmetry", symmetry_pipeline, ["symmetry_ord"]) 
#     ], 
#     remainder='passthrough',
#     n_jobs=-1)


 ## ChatGPT generated

In [14]:
ordinal_transformer = Pipeline(steps=[
    ('encoder', OrdinalEncoder(categories=[
        ['F', 'IF', 'VVS1', 'VVS2', 'VS1', 'VS2', 'SI1', 'SI2', 'SI3', 'I1', 'I2', 'I3'], # clarity_ord
        ['N', 'VS', 'S', 'M', 'SL', 'L', 'VL', 'EL', 'unknown'], # culet_size_ord
        ['Ideal', 'Excellent', 'Very Good', 'Good', 'Fair', 'None', 'unknown'], # cut_quality_ord
        ['Excellent', 'Very Good', 'Good', 'Fair', 'Poor'], # polish_ord
        ['Excellent', 'Very Good', 'Good', 'Fair', 'Poor'] # symmetry_ord
    ]))
])


In [15]:
nominal_transformer = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

In [16]:
# define the preprocessor with the transformers
preprocessor = ColumnTransformer(transformers=[
    ('ord', ordinal_transformer, ['clarity', 'culet_size', 'cut_quality', 'polish', 'symmetry']),
    ('nom', nominal_transformer, ['nominal_col1', 'nominal_col2', 'nominal_col3']),
], remainder='passthrough')

In [17]:
# define the model pipeline
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression())
])

In [18]:
# define a scorer for cross-validation
scorer = make_scorer(accuracy_score)

In [24]:
# load the dataset and split into training and test sets

df = pd.read_pickle("../data/diamonds.pkl")
X = df.drop('total_sales_price', axis=1)
y = df['total_sales_price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train.shape, y_train.shape, X_test.shape, y_test.shape

((175762, 24), (175762,), (43941, 24), (43941,))

In [22]:
# fit the model pipeline on the training data and evaluate using cross-validation
scores = cross_val_score(model, X_train, y_train, cv=5, scoring=scorer)


/opt/homebrew/Caskroom/miniforge/base/envs/supervised/lib/python3.9/site-packages/sklearn/model_selection/_split.py:676: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/supervised/lib/python3.9/site-packages/sklearn/model_selection/_validation.py:372: FitFailedWarning: 
5 fits failed out of a total of 5.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "/opt/homebrew/Caskroom/miniforge/base/envs/supervised/lib/python3.9/site-packages/pandas/core/indexes/base.py", line 3621, in get_loc
    return self._engine.get_loc(casted_key)
  File "pandas/_libs/inde

In [ ]:
# print the mean and standard deviation of the cross-validation scores
print("Accuracy: %0.2f (+/- %0.2f)" % (scores.mean(), scores.std() * 2))


## end ChatGPT

In [ ]:
# Define the model
model = LogisticRegression()

# Define the pipeline
pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('model', model)])

# Define the scoring
scoring = 'accuracy'

# Fit and evaluate the pipeline using cross-validation
scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring=scoring)


In [ ]:
model = LogisticRegression(random_state=0)

In [ ]:
df.dtypes


In [ ]:
model_pipeline = Pipeline(steps=[
    ('col_converter',  col_trans),
    ('model',  model)
])

model_pipeline  # click on the diagram below to see the details of each step

In [ ]:
model_pipeline.fit(X_train, y_train)

In [ ]:
print("model score: %.3f" % pipeline.score(X_test, y_test))

## Yet another attempt


In [ ]:
# numeric_features = numeric_columns, 
# nominal_features = nominal_columns,
# ordinal_features = ordinal_columns, 


numeric_transformer = Pipeline(
    steps=[(("scaler", StandardScaler())) ])


nominal_transformer = Pipeline(
    steps=[
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ("selector", SelectPercentile(chi2, percentile=50)) ])

           
           
encoder = OrdinalEncoder(categories=ordinal_columns)
          
ordinal_trasformer = Pipeline(
    steps=[
        ("encoder", OrdinalEncoder(handle_unknown="ignore")),
        ("selector", SelectPercentile(chi2, percentile=50)) ])

           
preprocessor = ColumnTransformer(
    transformers=[
        ("nominals", nominal_transformer, nominal_columns)
        ("ordinals", numeric_transformer, numerical_columns),
        
    ], remainder = 'passthough'
)



## end

In [ ]:
pipe = make_pipeline(transform_cols, LinearRegression())
pipe  # click on the diagram below to see the details of each step 


In [ ]:
model = pipe.fit(X_train, y_train)

In [ ]:
y_hat = pipe.predict(X_test)

In [ ]:
y_hat= pipe.predict(X_test)
print("RMSE: {}".format(np.sqrt(mean_squared_error((y_test),(y_hat)))))

In [ ]:
# pipeline = Pipeline(steps = [
#                ('preprocessor', preprocessor)
#               ,('regressor',RandomForestRegressor())
#            ])


In [ ]:
# pipe = make_pipeline(preprocessor, LinearRegression())
# pipe  # click on the diagram below to see the details of each steppipe = make_pipeline(preprocessor, LinearRegression())
# pipe  # click on the diagram below to see the details of each step

In [ ]:
# model = pipe.fit(X_train, y_train)

# y_pred = pipe.predict(X_train)

In [ ]:
# y_pred

In [ ]:
# numeric_features = numerical_features 
# numeric_transformer = Pipeline(
#     steps=[("scaler", StandardScaler())])

# categorical_features = categorical_features
# categorical_transformer = Pipeline(
#     steps=[("encoder", OneHotEncoder())])

# ordinal_features = ordinal_features
# ordinal_transformer = Pipeline(
#     steps=[("encoder", OrdinalEncoder())])


# preprocessor = ColumnTransformer(
#     transformers=[
#         ("cat", categorical_transformer, categorical_features),
#         ("ord", ordinal_transformer, ordinal_features ),
#         ("num", numeric_transformer, numeric_features)
#         ])

In [ ]:
predictions = model.predict(X_test)
print (f'Model r2 score:{r2_score(predictions, y_test)}')

In [ ]:
transformer = QuantileTransformer(output_distribution='normal')
regressor = LinearRegression()

In [ ]:
regr = TransformedTargetRegressor(regressor=regressor, transformer=transformer)

In [ ]:
#regr.fit(X_train, y_train)

In [ ]:
# print("model score: %.3f" % clf.score(X_test, y_test))

In [ ]:
# # 

# tree=DecisionTreeRegressor(max_depth=3)
# tree.fit(X_train, y_train)

# y_pred = tree.predict(X_test)



# print("RMSE: {}".format(np.sqrt(mean_squared_error((y_test),(y_pred)))))
# print("R2  : {}".format(np.sqrt(r2_score((y_test),(y_pred)))))

In [ ]:
# print(f"""
# Decision Trees has a {round((16057.1061-13921.2573)/16057.1061 *100 , 1)} % improvement over baseline in RMSE 
# and a {round((.9262440179416279-.8350244922620658)/.8350244922620658 *100 , 1)} % improvement in R2
# """)

In [ ]:
# xgb_classifier = xgb.XGBClassifier()

In [ ]:
# xgb_r = xgb.XGBRegressor(objective ='reg:squarederror', n_estimators = 1000, seed = 123, verbosity=1)
# xgb_r.fit(X_train, y_train)

# y_pred = xgb_r.predict(X_test)

In [ ]:
# print("RMSE: {}".format(np.sqrt(mean_squared_error((y_test),(y_pred)))))
# print("R2  : {}".format(np.sqrt(r2_score((y_test),(y_pred)))))

### XGBoost regression is strong for two reasons. Good performance and so much faster than Random Forrest Regression. And this is for a basically untuned model.

Grid Search didn't preform as well as expected. In fact it was substantially worse.

Yeah, I don't get that.
```
Fitting 5 folds for each of 54 candidates, totalling 270 fits
Best parameters: {'colsample_bytree': 0.7, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 1000}
Lowest RMSE:  25154.080799724477
```

Let's try a different set of parameters

Started at 5:30pm

```
params = {'learning_rate': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6],
          'n_estimators': [50, 100, 150, 200, 300],
          'colsample_bytree': [.5,  0.7, 0.8] }

clf = GridSearchCV(estimator=xgb_r, param_grid=params,
                   scoring='neg_mean_squared_error', verbose=1)

clf.fit(X, y)

print("Best parameters:", clf.best_params_)
print("Lowest RMSE: ", (-clf.best_score_)**(1/2.0))
```

Fitting 5 folds for each of 90 candidates, totalling 450 fits <br>
Best parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.6, 'n_estimators': 300} <br>
Lowest RMSE:  25282.458416182304

## Probably not going to be used

In [ ]:
# clarity_ord = [['F', 0], ['IF', 1], ['VVS1', 2], ['VVS2', 3], ['VS1', 4], ['VS2', 5],
#            ['SI1', 6], ['SI2',7], ['SI3',8], ['I1', 9], ['I2', 10], ['I3', 11]]

# #clarity_ord = [['F', 'IF', 'VVS1', 'VVS2', 'VS1', 'VS2', 'SI1', 'SI2', 'SI3', 'I1', 'I2', 'I3']]

# cut_quality_ord = [['Ideal', 0],['Excellent', 1], ['Very Good', 2], ['Good', 3],
#                ['Fair', 4],['None', 5],['unknown', 5]]

# symmetry_ord = [['Excellent', 0], ['Very Good', 1], ['Good', 2], ['Fair', 3], ['Poor', 4]]

# polish_ord = [['Excellent', 0], ['Very Good', 1], ['Good', 2], ['Fair', 3], ['Poor', 4]]

# culet_size_ord = [['N', 0], ['VS', 1], ['S', 2], ['M', 3],['SL', 4], ['L', 5], ['VL', 6],
#               ['EL', 7], ['unknown', 8]]

# ordinal_cols = [clarity_ord, cut_quality_ord, symmetry_ord, polish_ord, culet_size_ord]

In [ ]:
# as make_column_transformer instead of ColumnTransformer


# clarity_ord = [['F', 'IF', 'VVS1', 'VVS2', 'VS1', 'VS2', 'SI1',  'SI2', 'SI3', 'I1', 'I2','I3']]
# cut_quality_ord = [['Ideal', 'Excellent',  'Very Good', 'Good', 'Fair', 'None', 'unknown']]
# symmetry_ord = [['Excellent','Very Good', 'Good', 'Fair', 'Poor']]
# polish_ord = [['Excellent','Very Good', 'Good', 'Fair', 'Poor']]
# culet_size_ord = [['N', 'VS', 'S', 'M', 'SL', 'L', 'VL', 'EL', 'unknown']]


# transform_cols = make_column_transformer(
#     (OrdinalEncoder(categories=clarity_ord), ['clarity']),
#     (OrdinalEncoder(categories=cut_quality_ord), ['cut_quality']),
#     (OrdinalEncoder(categories=symmetry_ord), ['symmetry']),
#     (OrdinalEncoder(categories=polish_ord), ['polish']),
#     (OrdinalEncoder(categories=culet_size_ord), ['culet_size']),
#     (OneHotEncoder(sparse=False), categorical_columns),
#     remainder='passthrough'
# )


# transform_cols